In [ ]:
import pandas as pd
import numpy as np

dfflight = pd.read_csv('data/raw/c-flight-activity.csv')
dfloyalty = pd.read_csv('data/raw/c-loyalty-history.csv')

In [5]:
dfflight.shape

(405624, 10)

In [6]:
dfloyalty.shape

(16737, 16)

In [4]:
dfflight.sample(3)

,Loyalty Number,Year,Month,Flights Booked,Flights with Companions,Total Flights,Distance,Points Accumulated,Points Redeemed,Dollar Cost Points Redeemed
192523,521949,2017,10,0,0,0,0,0.0,0,0
210572,513123,2018,1,8,0,8,960,96.0,0,0
371203,967150,2018,10,4,4,8,792,79.0,0,0


In [5]:
dfloyalty.sample(3)

,Loyalty Number,Country,Province,City,Postal Code,Gender,Education,Salary,Marital Status,Loyalty Card,CLV,Enrollment Type,Enrollment Year,Enrollment Month,Cancellation Year,Cancellation Month
11380,842959,Canada,Ontario,Thunder Bay,K8T 5M5,Male,Bachelor,55296.0,Married,Star,2627.83,Standard,2015,12,NaN,NaN
14557,129328,Canada,Yukon,Whitehorse,Y2K 6R0,Female,Bachelor,76971.0,Married,Star,7379.46,Standard,2015,2,2015.0,10.0
889,660436,Canada,Ontario,Toronto,M2M 6J7,Male,College,NaN,Single,Aurora,5123.77,Standard,2017,5,NaN,NaN


In [2]:
#Vamos a proceder a trabajar con una copia 

df1 = dfflight.copy()
df2 = dfloyalty.copy()

In [9]:
df1.sample(3)

,Loyalty Number,Year,Month,Flights Booked,Flights with Companions,Total Flights,Distance,Points Accumulated,Points Redeemed,Dollar Cost Points Redeemed
204438,187835,2018,1,0,0,0,0,0.0,0,0
380579,567082,2018,11,0,0,0,0,0.0,0,0
91238,459971,2017,6,7,0,7,2016,201.0,0,0


In [10]:
df2.sample(3)

,Loyalty Number,Country,Province,City,Postal Code,Gender,Education,Salary,Marital Status,Loyalty Card,CLV,Enrollment Type,Enrollment Year,Enrollment Month,Cancellation Year,Cancellation Month
2771,769361,Canada,Saskatchewan,Regina,S6J 3G0,Male,Bachelor,76723.0,Married,Aurora,8828.38,Standard,2013,8,NaN,NaN
15080,969632,Canada,Quebec,Hull,J8Y 3Z5,Female,Bachelor,65935.0,Divorced,Star,8427.17,Standard,2013,2,NaN,NaN
16409,109478,Canada,Ontario,Toronto,M8Y 4K8,Male,Bachelor,84346.0,Married,Star,20998.26,Standard,2014,7,NaN,NaN


In [3]:
df1.columns = df1.columns.str.lower()
df2.columns = df2.columns.str.lower()


In [8]:
df2.sample()

,loyalty number,country,province,city,postal code,gender,education,salary,marital status,loyalty card,clv,enrollment type,enrollment year,enrollment month,cancellation year,cancellation month
582,413185,Canada,Nova Scotia,Halifax,B3C 2M8,Female,Doctor,175830.0,Married,Star,4792.51,Standard,2016,7,NaN,NaN


In [ ]:
def reemplazar_guion_bajo(df):
    """
    Reemplazamos los espacios en los nombres de columnas por guiones bajos usando un bucle.
    """
    nuevos_nombres = {}

    for col in df.columns:
        nuevos_nombres[col] = col.replace(' ', '_')

    df.rename(columns=nuevos_nombres, inplace=True)
    return df

In [5]:
df1 = reemplazar_guion_bajo(df1)
df2 = reemplazar_guion_bajo(df2)

In [6]:
print(df1.sample(3))
print(df2.sample(3))

        loyalty_number  year  month  flights_booked  flights_with_companions  \
195802          625524  2017     12               5                        5   
26639           617008  2017      2               0                        0   
285513          903359  2018      5               0                        0   

        total_flights  distance  points_accumulated  points_redeemed  \
195802             10       770                77.0                0   
26639               0         0                 0.0                0   
285513              0         0                 0.0                0   

        dollar_cost_points_redeemed  
195802                            0  
26639                             0  
285513                            0  
       loyalty_number country          province      city postal_code  gender  \
15147          664183  Canada  British Columbia  Whistler     V6T 1Y8    Male   
2523           885474  Canada           Ontario   Toronto     M8Y 4K8  Femal

In [7]:
df1 = df1.rename(columns={'loyalty_number': 'frequency_id'})

df1['frequency_id'].head()

0    100018
1    100102
2    100140
3    100214
4    100272
Name: frequency_id, dtype: int64

In [8]:
df1.insert(0, 'client_id', df1['frequency_id'])


In [9]:
df1.sample(3)

,client_id,frequency_id,year,month,flights_booked,flights_with_companions,total_flights,distance,points_accumulated,points_redeemed,dollar_cost_points_redeemed
379331,499875,499875,2018,11,1,0,1,645,64.0,0,0
264604,689839,689839,2018,4,0,0,0,0,0.0,0,0
25121,538902,538902,2017,2,0,0,0,0,0.0,0,0


In [10]:
df1.columns

Index(['client_id', 'frequency_id', 'year', 'month', 'flights_booked',
       'flights_with_companions', 'total_flights', 'distance',
       'points_accumulated', 'points_redeemed', 'dollar_cost_points_redeemed'],
      dtype='object')

In [11]:
# Agrupamos y sumamos valores numéricos
df1_1 = df1.groupby('client_id').agg({
    'frequency_id': 'count',
    'flights_booked': 'sum',
    'flights_with_companions': 'sum',
    'distance': 'sum',
    'points_accumulated': 'sum',
    'points_redeemed': 'sum',
    'dollar_cost_points_redeemed': 'sum',
    'year': ['min', 'max', lambda x: x.nunique()],
}).reset_index()

# Aplanar el MultiIndex
df1_1.columns = [
    'client_id',
    'frequency_count',
    'total_flights_booked',
    'total_flights_with_companions',
    'total_distance',
    'total_points_accumulated',
    'total_points_redeemed',
    'total_dollar_cost_redeemed',
    'first_year',
    'last_year',
    'active_years'
]

# Para manejar los meses creamos un diccionario para mapear números de mes como nombres de meses

month_names = {
    1: 'jan', 2: 'feb', 3: 'mar', 4: 'apr', 5: 'may', 6: 'jun',
    7: 'jul', 8: 'aug', 9: 'sep', 10: 'oct', 11: 'nov', 12: 'dec'
}

# Filtramos para incluir sólo meses con actividad real
df1_active = df1[(df1['flights_booked'] > 0) | (df1['distance'] > 0)]

# Sólo entonces listamos los meses únicos por cliente 
months_x_client = df1_active.groupby('client_id')['month'].unique().reset_index(name='months')

# Fragmentamos la columna 'months'
exploded = months_x_client.explode('months')

# Mapeamos los números como nombres después de fragmentar
exploded['month_names'] = exploded['months'].map(month_names)

# Creamos columnas dummy
dummies = pd.get_dummies(exploded['month_names'], prefix='month')

# Agrupamos y sumamos
months_dummy = dummies.groupby(exploded['client_id']).sum()

# Lo hacemos efectivo
df1_1 = df1_1.merge(months_dummy, on='client_id', how='left')

# Rellenamos el NaN que nos surge en las columnas sin actividad con 0 en las columnas de meses
month_cols = [col for col in df1_1.columns if col.startswith('month_')]
df1_1[month_cols] = df1_1[month_cols].fillna(0)




In [12]:
print(months_x_client.head(10))


   client_id                                   months
0     100018     [1, 2, 10, 4, 6, 7, 9, 8, 11, 12, 3]
1     100102  [1, 4, 12, 8, 9, 10, 11, 2, 5, 6, 7, 3]
2     100140     [1, 3, 4, 5, 6, 7, 8, 9, 11, 12, 10]
3     100214              [2, 6, 8, 9, 10, 12, 3, 11]
4     100272  [2, 4, 6, 7, 3, 8, 11, 5, 12, 1, 9, 10]
5     100301     [2, 3, 5, 6, 7, 8, 9, 10, 11, 12, 1]
6     100364         [2, 5, 6, 7, 8, 10, 12, 1, 9, 4]
7     100380     [2, 4, 5, 6, 8, 3, 10, 1, 9, 11, 12]
8     100428     [1, 11, 2, 5, 8, 9, 12, 3, 6, 7, 10]
9     100504                 [8, 10, 11, 12, 1, 2, 3]


In [13]:
df1_1['month_apr'].unique()

array([1., 0.])

In [14]:
df1_1.head()

,client_id,frequency_count,total_flights_booked,total_flights_with_companions,total_distance,total_points_accumulated,total_points_redeemed,total_dollar_cost_redeemed,first_year,last_year,...,month_dec,month_feb,month_jan,month_jul,month_jun,month_mar,month_may,month_nov,month_oct,month_sep
0,100018,24,157,35,50682,5376.00,1513,123,2017,2018,...,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0
1,100102,24,173,42,40222,4115.25,1195,96,2017,2018,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
2,100140,24,152,38,41252,4184.25,593,48,2017,2018,...,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
3,100214,24,79,17,33982,3426.00,861,70,2017,2018,...,1.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
4,100272,24,127,36,40872,4108.04,1007,82,2017,2018,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


In [15]:
df1_1.shape

(16737, 23)

In [16]:
df2.shape

(16737, 16)

In [17]:
df1_1.rename(columns={'client_id': 'loyalty_number'}, inplace=True)

In [18]:
df_merged = pd.merge(
    df2,                  # DataFrame 2 con ID único de clientes en común
    df1_1,                # DataFrame 1
    on='loyalty_number',  # Columna común (ID)
    how='left'            
)


In [19]:
df_merged.shape

(16737, 38)

In [20]:
df_merged.columns

Index(['loyalty_number', 'country', 'province', 'city', 'postal_code',
       'gender', 'education', 'salary', 'marital_status', 'loyalty_card',
       'clv', 'enrollment_type', 'enrollment_year', 'enrollment_month',
       'cancellation_year', 'cancellation_month', 'frequency_count',
       'total_flights_booked', 'total_flights_with_companions',
       'total_distance', 'total_points_accumulated', 'total_points_redeemed',
       'total_dollar_cost_redeemed', 'first_year', 'last_year', 'active_years',
       'month_apr', 'month_aug', 'month_dec', 'month_feb', 'month_jan',
       'month_jul', 'month_jun', 'month_mar', 'month_may', 'month_nov',
       'month_oct', 'month_sep'],
      dtype='object')

In [ ]:
df_merged.to_csv('data/processed/flight-loyalty-activity.csv', index=False)


In [ ]:
df_total = pd.read_csv('data/processed/flight-loyalty-activity.csv')

In [41]:
df_total

,loyalty_number,country,province,city,postal_code,gender,education,salary,marital_status,loyalty_card,...,month_dec,month_feb,month_jan,month_jul,month_jun,month_mar,month_may,month_nov,month_oct,month_sep
0,480934,Canada,Ontario,Toronto,M2Z 4K1,Female,Bachelor,83236.0,Married,Star,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1,549612,Canada,Alberta,Edmonton,T3G 6Y6,Male,College,NaN,Divorced,Star,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
2,429460,Canada,British Columbia,Vancouver,V6E 3D9,Male,College,NaN,Single,Star,...,1.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0
3,608370,Canada,Ontario,Toronto,P1W 1K4,Male,College,NaN,Single,Star,...,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0
4,530508,Canada,Quebec,Hull,J8Y 3Z5,Male,Bachelor,103495.0,Married,Star,...,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16732,823768,Canada,British Columbia,Vancouver,V6E 3Z3,Female,College,NaN,Married,Star,...,1.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,1.0
16733,680886,Canada,Saskatchewan,Regina,S1J 3C5,Female,Bachelor,89210.0,Married,Star,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
16734,776187,Canada,British Columbia,Vancouver,V5R 1W3,Male,College,NaN,Single,Star,...,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
16735,906428,Canada,Yukon,Whitehorse,Y2K 6R0,Male,Bachelor,-57297.0,Married,Star,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0


## Gestión de nulos

In [42]:
df_total['salary'] = df_total['salary'].abs()
df_total['salary'].unique()



#Había por ahí valores negativos

array([ 83236.,     nan, 103495., ...,  76178.,  91970.,  57297.],
      shape=(5891,))

In [43]:
df_total['salary'].isnull().sum() 
# no es que podamos hacer mucho con la información que no nos han proporcionado, podríamos calcular una mediana según nivel de estudios
# ya que hemos comprobado que los NaN se encuentran asociados a 'college'

np.int64(4238)

In [44]:
# En las previsualizaciones hemos observado que se deben todos al registro 'college'
# Así que realizamos la mediana de salary para 'bachelor'

# Calcula la mediana de salary para 'bachelor'
median_bachelor_salary = df_total[df_total['education'] == 'Bachelor']['salary'].median()

# Creamos una máscara para seleccionar solo los valores nulos en 'salary' donde 'education' es 'college'
mask = (df_total['education'] == 'College') & (df_total['salary'].isna())

# Aplica fillna() solo a las filas que cumplen la máscara
df_total.loc[mask, 'salary'] = df_total.loc[mask, 'salary'].fillna(median_bachelor_salary)



In [45]:
df_total.isnull().sum()

loyalty_number                       0
country                              0
province                             0
city                                 0
postal_code                          0
gender                               0
education                            0
salary                               0
marital_status                       0
loyalty_card                         0
clv                                  0
enrollment_type                      0
enrollment_year                      0
enrollment_month                     0
cancellation_year                14670
cancellation_month               14670
frequency_count                      0
total_flights_booked                 0
total_flights_with_companions        0
total_distance                       0
total_points_accumulated             0
total_points_redeemed                0
total_dollar_cost_redeemed           0
first_year                           0
last_year                            0
active_years             

In [ ]:
# cancellation_year = aprovechamos los nulos para crear una columna de validación llamada 'active_member'

df_total['active_member'] = df_total['cancellation_year'].isna()


In [47]:
df_total.columns.tolist()

['loyalty_number',
 'country',
 'province',
 'city',
 'postal_code',
 'gender',
 'education',
 'salary',
 'marital_status',
 'loyalty_card',
 'clv',
 'enrollment_type',
 'enrollment_year',
 'enrollment_month',
 'cancellation_year',
 'cancellation_month',
 'frequency_count',
 'total_flights_booked',
 'total_flights_with_companions',
 'total_distance',
 'total_points_accumulated',
 'total_points_redeemed',
 'total_dollar_cost_redeemed',
 'first_year',
 'last_year',
 'active_years',
 'month_apr',
 'month_aug',
 'month_dec',
 'month_feb',
 'month_jan',
 'month_jul',
 'month_jun',
 'month_mar',
 'month_may',
 'month_nov',
 'month_oct',
 'month_sep',
 'active_member']

In [48]:
df_total.isnull().sum()

loyalty_number                       0
country                              0
province                             0
city                                 0
postal_code                          0
gender                               0
education                            0
salary                               0
marital_status                       0
loyalty_card                         0
clv                                  0
enrollment_type                      0
enrollment_year                      0
enrollment_month                     0
cancellation_year                14670
cancellation_month               14670
frequency_count                      0
total_flights_booked                 0
total_flights_with_companions        0
total_distance                       0
total_points_accumulated             0
total_points_redeemed                0
total_dollar_cost_redeemed           0
first_year                           0
last_year                            0
active_years             

In [ ]:
# Dejaremos los nulos tal cual están, no implican mayor sesgo
# la columna active_member nos valida que, en efecto, no son errores

In [ ]:
df_total.to_csv('data/processed/flight-loyalty-activity.csv', index=False)